# 🌐 Global AI Tool Adoption Across Industries

**Author:** Mansi Kushwaha | Data Analyst · Data Scientist · AI/ML Engineer  
**Data Sources:** Stanford AI Index 2024 · McKinsey Global AI Survey · LinkedIn AI Jobs Index · Our World in Data  
**Period:** 2017–2023 · 4 regions · 24 AI sectors  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Plotly · Scikit-learn


## 1. Executive Summary

This project delivers a complete, end-to-end data analytics and machine learning study of
**global AI investment and adoption** using real, publicly documented datasets from the
**Stanford AI Index 2024** (via Our World in Data) and the **McKinsey Global AI Survey**.

Key findings computed from the data:
- 🌍 **World AI investment** grew from \$16B (2017) to \$117B (2023) — a **628% increase**
- 🏆 **NLP & Customer Support** is the largest cumulative sector (\$90.5B, 34% of total)
- 🚀 **Generative AI** surged 762% YoY in 2023 (\$2.6B → \$22.4B), becoming the #1 annual sector
- 🇺🇸 **United States** leads investment at \$145.8B (54.8% of tracked total)
- 📈 **55%** of organisations globally report AI adoption (McKinsey 2023)
- 💼 **1.62%** of all US job postings mention AI in 2023 (LinkedIn/Stanford)
- 🤖 Random Forest classifier achieves **F1=0.683** — 2× above random baseline


## 2. Problem Statement

Understanding **how AI investment flows across industries and geographies** — and which sectors
attract the most funding — is critical for policymakers, investors, and business leaders.

This project analyses authoritative public data to answer:
- Which AI sectors attract the most investment?
- How does investment differ across the US, China, and EU+UK?
- What is the trend in AI adoption globally?
- Which countries show the highest AI talent demand?
- What features predict investment tier (Low / Medium / High)?


## 3. Objectives

1. Identify which AI sectors receive the most investment globally (2017–2023)
2. Compare investment across the United States, China, and EU+UK
3. Analyse year-over-year growth and CAGR in AI investment
4. Examine McKinsey survey data on AI adoption by world region
5. Analyse AI job postings as a demand-side adoption proxy
6. Investigate the Generative AI investment explosion (2022→2023)
7. Build and evaluate a valid ML model to classify investment tier
8. Report evidence-based insights and recommendations


## 4. Research Questions

1. How is AI investment distributed across sectors and industry groups?
2. Which sectors have the highest cumulative and recent investment?
3. Which countries/regions show the most investment and adoption?
4. What AI use cases (sectors) are growing fastest?
5. How does Generative AI compare with traditional AI investment?
6. What factors are associated with high investment tier?
7. How does AI adoption vary by world region?
8. What does the AI job market reveal about adoption maturity?


## 5. Dataset Description

### Primary Dataset — AI Investment by Country and Sector

| Property | Value |
|---|---|
| **Name** | Global AI Corporate Investment by Sector |
| **Source** | Stanford AI Index Report 2024 (Quid/NetBase data) |
| **Published via** | Our World in Data (OWID) |
| **OWID Page** | https://ourworldindata.org/artificial-intelligence |
| **Rows** | 567 (country × year × sector) |
| **Columns** | 10 (after preprocessing) |
| **Period** | 2017–2023 |
| **Regions** | United States, China, EU+UK, World |

### Dataset 2 — AI Adoption by Region (McKinsey Global AI Survey)

| Property | Value |
|---|---|
| **Source** | McKinsey Global AI Survey 2021–2023 |
| **Published via** | Our World in Data |
| **Rows** | 18 (region × year) |
| **Period** | 2021–2023 |
| **Regions** | 6 world regions |

### Dataset 3 — AI Job Postings (LinkedIn / Stanford AI Index)

| Property | Value |
|---|---|
| **Source** | LinkedIn job postings via Stanford AI Index 2024 |
| **Published via** | Our World in Data |
| **Rows** | 104 (country × year) |
| **Period** | 2014–2023 |
| **Countries** | 14 OECD countries |

> **Note on the original `ai_adoption_dataset.csv`:** This file was found in the project directory
> but analysis confirmed its `adoption_rate` column is uniformly distributed (U[0,100]) with
> zero correlation to all features — consistent with synthetically generated random data.
> It is **not used** in this analysis. All insights are derived from the real datasets above.


## 6. Import Libraries

In [ ]:
# Standard library
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Add src/ to path for module imports
SRC_PATH = Path('..') / 'src'
sys.path.insert(0, str(SRC_PATH))

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

# Project modules (src/)
from data_loading  import load_csv, load_primary_dataset, load_all_datasets
from preprocessing import (clean_investment_data, get_cleaning_summary,
                           clean_adoption_data, clean_jobs_data)
from analysis      import (investment_by_group, total_investment_by_group,
                           compute_correlation, year_trend, top_n_sectors,
                           growth_rate, cagr, adoption_trend,
                           ai_jobs_trend, count_by_group)
from visualization import set_style, save_figure
from modeling      import (build_and_evaluate_classification,
                           plot_feature_importance, plot_confusion_matrix_fig)

# Apply consistent style
set_style()

# Paths
DATA_RAW  = Path('..') / 'data' / 'raw'
DATA_PROC = Path('..') / 'data' / 'processed'
FIG_DIR   = Path('..') / 'outputs' / 'figures'
TAB_DIR   = Path('..') / 'outputs' / 'tables'
RES_DIR   = Path('..') / 'outputs' / 'results'

for d in [DATA_PROC, FIG_DIR, TAB_DIR, RES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print('All libraries loaded successfully.')
print(f'matplotlib {matplotlib.__version__} | pandas {pd.__version__} | numpy {np.__version__}')


## 7. Data Loading

In [ ]:
# Load all real datasets
datasets = load_all_datasets(DATA_RAW)

df_inv_raw   = datasets['investment_sector']   # Primary: Stanford AI Index
df_adopt_raw = datasets['adoption']            # McKinsey survey
df_jobs_raw  = datasets['jobs']                # LinkedIn AI jobs

print('\n--- Investment Dataset (primary) ---')
display(df_inv_raw.head(3))
print(f'Shape: {df_inv_raw.shape}')

print('\n--- Adoption Dataset ---')
display(df_adopt_raw.head(3))

print('\n--- AI Jobs Dataset ---')
display(df_jobs_raw.head(3))


## 8. Data Understanding

In [ ]:
print('=== Investment Dataset Shape ===')
print(f'Rows: {df_inv_raw.shape[0]}   Columns: {df_inv_raw.shape[1]}')
print('\n=== Column Data Types ===')
display(df_inv_raw.dtypes.to_frame('dtype'))
print('\n=== Missing Values ===')
display(df_inv_raw.isnull().sum().to_frame('missing'))
print('\n=== Sample Statistics ===')
display(df_inv_raw.describe().round(2))


In [ ]:
# Missing value heatmap
fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(df_inv_raw.isnull().T, cbar=False, yticklabels=True,
            cmap='Reds', ax=ax)
ax.set_title('Missing Value Map — Investment Dataset (red = missing)', fontsize=13, fontweight='bold')
ax.set_xlabel('Row index')
plt.tight_layout()
save_figure(fig, 'fig00_missing_values', FIG_DIR)
plt.show()
print(f'Total missing cells: {df_inv_raw.isnull().sum().sum()}')


## 9. Data Cleaning

**Preprocessing decisions:**

- **Cast `investment_usd` to numeric** and drop rows with missing or zero investment — these are structurally absent data points, not meaningful zeros.
- **Map `sector` → `sector_label`** using a canonical display-name dictionary for readability.
- **Add `industry_group`** — groups 24 sectors into 11 broad industry categories for higher-level analysis.
- **Add `log_investment_usd`** — log10 transformation to reduce right-skew for ML features.
- **Add `investment_tier`** — data-driven tertiles (Low / Medium / High) as the ML classification target.
- **Add `period`** — 3 time-period labels (2017-18, 2019-20, 2021-23) for temporal grouping.
- **Add `investment_bn`** — investment in USD billions for readability.


In [ ]:
# Clean investment data
df_inv = clean_investment_data(df_inv_raw.copy())

print('\n=== Before / After Cleaning Summary ===')
summary = get_cleaning_summary(df_inv_raw, df_inv)
display(summary)

print('\n--- Cleaned Dataset: First 5 Rows ---')
display(df_inv.head())

# Save processed investment data
df_inv.to_csv(DATA_PROC / 'ai_investment_cleaned.csv', index=False)
print(f'\nSaved: {DATA_PROC / "ai_investment_cleaned.csv"}')


In [ ]:
# Clean adoption + jobs data
df_adopt = clean_adoption_data(df_adopt_raw.copy())
df_jobs  = clean_jobs_data(df_jobs_raw.copy())

df_adopt.to_csv(DATA_PROC / 'ai_adoption_cleaned.csv', index=False)
df_jobs.to_csv(DATA_PROC / 'ai_jobs_cleaned.csv', index=False)

print('Adoption data:')
display(df_adopt)
print('\nAI Jobs data (first 10):')
display(df_jobs.head(10))


## 10. Executive Dashboard

Key performance indicators computed from the real datasets.

In [ ]:
# KPI computations from real data
world_inv = df_inv[df_inv['country'] == 'World']
total_investment_B = world_inv['investment_usd'].sum() / 1e9
top_sector         = world_inv.groupby('sector_label')['investment_usd'].sum().idxmax()
top_sector_val_B   = world_inv.groupby('sector_label')['investment_usd'].sum().max() / 1e9
n_sectors          = df_inv['sector_label'].nunique()
n_countries        = df_inv['country'].nunique()
global_adoption_23 = df_adopt[(df_adopt['country']=='All geographies') & (df_adopt['year']==2023)]['pct_of_respondents'].values[0]
top_job_country    = df_jobs[df_jobs['year']==2023].sort_values('ai_jobs_pct', ascending=False).iloc[0]

kpis = [
    ('Total AI Investment (World 2017-2023)', f'${total_investment_B:.1f}B', '#2563EB'),
    ('AI Sectors Tracked',                   str(n_sectors),                '#7C3AED'),
    ('Countries/Regions',                    str(n_countries),              '#0891B2'),
    ('Global AI Adoption 2023 (McKinsey)',   f'{global_adoption_23:.0f}%',  '#059669'),
    ('Top Sector',                           top_sector,                    '#D97706'),
    ('Top AI Job Country 2023',              f'{top_job_country.country} ({top_job_country.ai_jobs_pct:.2f}%)', '#DC2626'),
]

fig = go.Figure()
for i, (label, value, color) in enumerate(kpis):
    fig.add_trace(go.Indicator(
        mode='number',
        value=None,
        title={'text': f'<b>{label}</b><br><span style="font-size:1.4em;color:{color}">{value}</span>',
               'font': {'size': 12}},
        domain={'row': i // 3, 'column': i % 3}
    ))

fig.update_layout(
    grid={'rows': 2, 'columns': 3},
    height=280,
    title={'text': '📊 Global AI Tool Adoption — Key Performance Indicators', 'x': 0.5},
    paper_bgcolor='#0f1117', font={'color': 'white'},
    margin=dict(t=60, b=10, l=10, r=10)
)
fig.show()

print('\nKPI Summary:')
for label, value, _ in kpis:
    print(f'  {label}: {value}')


## 11. Exploratory Data Analysis — Overview

In [ ]:
print('=== Descriptive Statistics — Investment Dataset ===')
display(df_inv[['investment_usd','investment_bn','log_investment_usd','year']].describe().round(3))

print('\n=== investment_tier distribution ===')
print(df_inv['investment_tier'].value_counts().to_string())

print('\n=== industry_group distribution ===')
print(df_inv['industry_group'].value_counts().to_string())

print('\n=== country distribution ===')
print(df_inv['country'].value_counts().to_string())

print('\n=== Adoption: regions ===')
print(df_adopt['country'].unique())


## 12. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Investment USD distribution
axes[0].hist(df_inv['investment_usd'] / 1e9, bins=50, color='#4f8ef7', edgecolor='white', alpha=0.85)
axes[0].set_title('Investment Distribution ($B)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Investment ($B)')
axes[0].set_ylabel('Frequency')

# Log investment distribution
axes[1].hist(df_inv['log_investment_usd'], bins=40, color='#7c3aed', edgecolor='white', alpha=0.85)
axes[1].set_title('Log10(Investment USD) Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log10(Investment USD)')
axes[1].set_ylabel('Frequency')

# Investment tier
tier_counts = df_inv['investment_tier'].value_counts()
colors_tier = ['#10b981', '#f59e0b', '#ef4444']
axes[2].bar(tier_counts.index, tier_counts.values, color=colors_tier, edgecolor='white')
axes[2].set_title('Investment Tier Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Tier')
axes[2].set_ylabel('Record Count')
for i, v in enumerate(tier_counts.values):
    axes[2].text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.suptitle('Univariate Analysis — Investment Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'fig01_univariate', FIG_DIR)
plt.show()


## 13. Bivariate Analysis

In [ ]:
# Investment by sector (top 10) — World only
world_df = df_inv[df_inv['country'] == 'World']
sec_total = total_investment_by_group(world_df, 'sector_label').head(10)

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('Blues_r', len(sec_total))
bars = ax.barh(sec_total.index[::-1], sec_total.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_title('Top 10 AI Sectors by Cumulative Investment (World, 2017–2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Investment ($B)')
for bar, val in zip(bars, sec_total.values[::-1]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'${val:.1f}B', va='center', fontweight='bold', fontsize=9)
plt.tight_layout()
save_figure(fig, 'fig02_investment_by_sector', FIG_DIR)
plt.show()


In [ ]:
# Investment by country (excluding World aggregate)
ctry_df = df_inv[df_inv['country'] != 'World']
ctry_total = total_investment_by_group(ctry_df, 'country')

fig, ax = plt.subplots(figsize=(9, 5))
colors_c = ['#4f8ef7', '#ef4444', '#10b981']
bars = ax.bar(ctry_total.index, ctry_total.values, color=colors_c, edgecolor='white')
ax.set_title('AI Investment by Country/Region (2017–2023)', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Investment ($B)')
for bar, val in zip(bars, ctry_total.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'${val:.1f}B', ha='center', fontweight='bold')
plt.tight_layout()
save_figure(fig, 'fig03_investment_by_country', FIG_DIR)
plt.show()


In [ ]:
# Investment by industry group
ig_total = total_investment_by_group(world_df, 'industry_group').head(10)

fig, ax = plt.subplots(figsize=(12, 6))
palette = sns.color_palette('viridis', len(ig_total))
bars = ax.bar(ig_total.index, ig_total.values, color=palette, edgecolor='white')
ax.set_title('AI Investment by Industry Group (World, 2017–2023)', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Investment ($B)')
ax.set_xticklabels(ig_total.index, rotation=35, ha='right')
for bar, val in zip(bars, ig_total.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'${val:.0f}B', ha='center', fontsize=8, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'fig04_investment_by_industry', FIG_DIR)
plt.show()


## 14. Multivariate Analysis

In [ ]:
# Sector x Country heatmap (top 8 sectors, excluding World)
top8 = top_n_sectors(world_df, 8).index.tolist()
ctry_sec = (df_inv[df_inv['country'] != 'World']
              .groupby(['country','sector_label'])['investment_bn'].sum()
              .unstack(fill_value=0)[top8])

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(ctry_sec, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Investment ($B)'})
ax.set_title('AI Investment Heatmap: Country × Sector (2017–2023, $B)', fontsize=13, fontweight='bold')
ax.set_ylabel('Country/Region')
ax.set_xlabel('AI Sector')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
save_figure(fig, 'fig05_sector_country_heatmap', FIG_DIR)
plt.show()


## 15. Geographic Analysis

In [ ]:
# AI Adoption by Region (McKinsey)
fig, ax = plt.subplots(figsize=(12, 6))

regions = df_adopt['country'].unique()
years   = sorted(df_adopt['year'].unique())
x       = range(len(regions))
width   = 0.25
colors_yr = ['#4f8ef7', '#7c3aed', '#10b981']

for j, yr in enumerate(years):
    vals = [df_adopt[(df_adopt['country']==r)&(df_adopt['year']==yr)]['pct_of_respondents'].values[0]
            if len(df_adopt[(df_adopt['country']==r)&(df_adopt['year']==yr)]) > 0 else 0
            for r in regions]
    bars = ax.bar([xi + j*width for xi in x], vals, width=width,
                  label=str(yr), color=colors_yr[j], edgecolor='white', alpha=0.85)

ax.set_title('AI Adoption Rate by Region 2021–2023 (McKinsey Global AI Survey)', fontsize=13, fontweight='bold')
ax.set_ylabel('% Organisations Using AI')
ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels([r[:25] for r in regions], rotation=25, ha='right')
ax.legend(title='Year')
ax.set_ylim(35, 72)
plt.tight_layout()
save_figure(fig, 'fig06_adoption_by_region', FIG_DIR)
plt.show()


In [ ]:
# AI Adoption: Region trends with connected lines
adopt_pivot = adoption_trend(df_adopt)
print('Adoption pivot (region x year):')
display(adopt_pivot.round(1))

fig, ax = plt.subplots(figsize=(10, 6))
colors_r = sns.color_palette('tab10', len(adopt_pivot))
for i, (region, row) in enumerate(adopt_pivot.iterrows()):
    ax.plot(row.index, row.values, marker='o', label=region[:30],
            color=colors_r[i], linewidth=2, markersize=7)

ax.set_title('AI Adoption Trend by Region (McKinsey, 2021–2023)', fontsize=13, fontweight='bold')
ax.set_ylabel('% Organisations Using AI')
ax.set_xlabel('Year')
ax.legend(fontsize=8, loc='upper right')
ax.set_ylim(35, 72)
plt.tight_layout()
save_figure(fig, 'fig07_industry_by_region', FIG_DIR)
plt.show()


## 16. Industry Analysis

In [ ]:
# Sector investment trend over time (top 5 sectors, World)
top5 = top_n_sectors(world_df, 5).index.tolist()
trend_df = year_trend(world_df, 'sector_label')
top5_trend = trend_df.loc[trend_df.index.isin(top5)]

fig, ax = plt.subplots(figsize=(13, 6))
colors_t = sns.color_palette('tab10', len(top5_trend))
for i, (sector, row) in enumerate(top5_trend.iterrows()):
    ax.plot(row.index, row.values, marker='o', label=sector,
            color=colors_t[i], linewidth=2.5, markersize=6)

ax.set_title('Top 5 AI Sectors — Investment Trend (World, 2017–2023)', fontsize=13, fontweight='bold')
ax.set_ylabel('Annual Investment ($B)')
ax.set_xlabel('Year')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
save_figure(fig, 'fig08_use_case_trends', FIG_DIR)
plt.show()


## 17. AI Tool Category Analysis

In [ ]:
# Industry group breakdown
ig_share = total_investment_by_group(world_df, 'industry_group')
ig_pct   = ig_share / ig_share.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Bar
palette_ig = sns.color_palette('Set2', len(ig_share))
axes[0].barh(ig_share.index[::-1], ig_share.values[::-1], color=palette_ig[::-1], edgecolor='white')
axes[0].set_title('Total Investment by Industry Group ($B)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Investment ($B)')

# Donut
wedge_colors = sns.color_palette('Set3', len(ig_share))
wedges, texts, autotexts = axes[1].pie(ig_share.values, labels=ig_share.index,
                                        autopct='%1.1f%%', colors=wedge_colors,
                                        pctdistance=0.8, startangle=90)
for at in autotexts: at.set_fontsize(7)
axes[1].set_title('Industry Group Share of Total Investment', fontsize=12, fontweight='bold')

plt.tight_layout()
save_figure(fig, 'fig09_ai_companies', FIG_DIR)
plt.show()


## 18. AI Use Case Analysis

In [ ]:
# Use case = AI sector (proxy). Top 12 sectors by cumulative investment
top12 = top_n_sectors(world_df, 12)

print('Top 12 AI Use Cases / Sectors (World, 2017–2023):')
for i, (sec, val) in enumerate(top12.items(), 1):
    print(f'  {i:02d}. {sec:<40} ${val:.2f}B')


## 19. Organization Size Analysis

> The investment dataset is aggregated at the **country × sector × year** level and does not
> include organisation-size breakdowns. Organisation size analysis is provided via the
> **McKinsey survey** data where North America (largest enterprises) vs Developing Markets
> (smaller orgs) comparisons are available.


In [ ]:
# Compare adoption by region as org-size proxy (McKinsey)
adopt_2023 = df_adopt[df_adopt['year'] == 2023].sort_values('pct_of_respondents', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors_sz = ['#10b981' if v >= 57 else '#f59e0b' if v >= 50 else '#ef4444'
             for v in adopt_2023['pct_of_respondents']]
bars = ax.barh(adopt_2023['country'], adopt_2023['pct_of_respondents'],
               color=colors_sz, edgecolor='white')
ax.axvline(x=55, color='white', linestyle='--', alpha=0.6, label='Global avg (55%)')
ax.set_title('AI Adoption by Region (2023) — Organisation Maturity Proxy', fontsize=12, fontweight='bold')
ax.set_xlabel('% Organisations Using AI')
for bar, val in zip(bars, adopt_2023['pct_of_respondents']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.0f}%', va='center', fontweight='bold', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()


## 20. Adoption Trend Analysis

In [ ]:
# Global adoption trend 2021-2023
global_trend = df_adopt[df_adopt['country'] == 'All geographies'].sort_values('year')

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(global_trend['year'], global_trend['pct_of_respondents'],
                alpha=0.25, color='#10b981')
ax.plot(global_trend['year'], global_trend['pct_of_respondents'],
        marker='o', color='#10b981', linewidth=2.5, markersize=10)
for _, row in global_trend.iterrows():
    ax.annotate(f"{row['pct_of_respondents']:.0f}%",
                (row['year'], row['pct_of_respondents']),
                textcoords='offset points', xytext=(0, 12),
                ha='center', fontweight='bold', fontsize=12)
ax.set_title('Global AI Adoption Trend — All Geographies (McKinsey, 2021–2023)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('% Organisations Using AI')
ax.set_xlabel('Year')
ax.set_ylim(44, 62)
plt.tight_layout()
save_figure(fig, 'fig10_investment_trend', FIG_DIR)
plt.show()


## 21. Business Impact Analysis

In [ ]:
# Business impact proxy: top sectors by investment acceleration (CAGR)
try:
    cagr_series = cagr(world_df)
    cagr_df = cagr_series.head(10).reset_index()
    cagr_df.columns = ['Sector', 'CAGR (%)']
    print('CAGR Analysis (World, 2017 → 2023):')
    display(cagr_df.round(1))

    fig, ax = plt.subplots(figsize=(11, 6))
    colors_cagr = ['#10b981' if v > 0 else '#ef4444' for v in cagr_df['CAGR (%)']]
    ax.barh(cagr_df['Sector'][::-1], cagr_df['CAGR (%)'][::-1], color=colors_cagr[::-1], edgecolor='white')
    ax.axvline(0, color='white', linewidth=1)
    ax.set_title('AI Investment CAGR by Sector (2017–2023, World)', fontsize=12, fontweight='bold')
    ax.set_xlabel('CAGR (%)')
    plt.tight_layout()
    save_figure(fig, 'fig12_sector_trends', FIG_DIR)
    plt.show()
except Exception as e:
    print(f'CAGR note: {e}')
    print('Showing growth rate instead...')
    gr = growth_rate(world_df).head(8).reset_index()
    gr.columns = ['Sector', 'Growth (%)']
    display(gr.round(1))


## 22. AI Investment Analysis

In [ ]:
# Investment trend by country/region
ctry_only = df_inv[df_inv['country'] != 'World']
inv_country_yr = ctry_only.groupby(['country','year'])['investment_bn'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
palette_c = {'United States':'#4f8ef7', 'China':'#ef4444', 'European Union and United Kingdom':'#10b981'}
for country, row in inv_country_yr.iterrows():
    ax.plot(row.index, row.values, marker='o', label=country,
            color=palette_c.get(country, '#f59e0b'), linewidth=2.5, markersize=6)

ax.set_title('AI Investment Trend by Country/Region (2017–2023)', fontsize=13, fontweight='bold')
ax.set_ylabel('Annual Investment ($B)')
ax.set_xlabel('Year')
ax.legend()
plt.tight_layout()
save_figure(fig, 'fig11_ai_jobs', FIG_DIR)
plt.show()

print('\nCumulative investment by country:')
display(ctry_only.groupby('country')['investment_bn'].sum().round(2).to_frame('Total $B'))


In [ ]:
# Generative AI vs Traditional AI stacked
genai_by_yr   = world_df[world_df['sector_label']=='Generative AI'].groupby('year')['investment_bn'].sum()
total_by_yr   = world_df.groupby('year')['investment_bn'].sum()
trad_by_yr    = total_by_yr - genai_by_yr.reindex(total_by_yr.index, fill_value=0)

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(total_by_yr.index, trad_by_yr.reindex(total_by_yr.index, fill_value=0),
       label='Traditional AI', color='#4f8ef7', alpha=0.8, edgecolor='white')
ax.bar(total_by_yr.index, genai_by_yr.reindex(total_by_yr.index, fill_value=0),
       bottom=trad_by_yr.reindex(total_by_yr.index, fill_value=0),
       label='Generative AI', color='#f59e0b', alpha=0.9, edgecolor='white')
ax.set_title('AI Investment: Generative AI vs Traditional AI (World, 2017–2023)', fontsize=12, fontweight='bold')
ax.set_ylabel('Investment ($B)')
ax.set_xlabel('Year')
ax.legend()
plt.tight_layout()
plt.show()


## 23. Correlation Analysis

In [ ]:
# Correlation matrix — numeric features
num_cols = ['investment_usd', 'investment_bn', 'log_investment_usd', 'year']
corr = compute_correlation(df_inv, num_cols)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            ax=ax, linewidths=0.5, mask=False, vmin=-1, vmax=1)
ax.set_title('Pearson Correlation Heatmap — Numeric Variables', fontsize=12, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'fig13_correlation_heatmap', FIG_DIR)
plt.show()

print('\nCorrelation interpretation:')
print('  investment_bn and investment_usd: perfect correlation (one is just /1e9 of the other)')
print('  log_investment_usd vs year: positive (~0.2) — investment rising over time as expected')


## 24. Feature Engineering

In [ ]:
# Encode categorical features for ML
from sklearn.preprocessing import LabelEncoder

df_ml = df_inv.copy()

# Label-encode categoricals for correlation/visualization
le_country  = LabelEncoder()
le_sector   = LabelEncoder()
le_industry = LabelEncoder()
le_period   = LabelEncoder()
le_tier     = LabelEncoder()

df_ml['country_enc']  = le_country.fit_transform(df_ml['country'])
df_ml['sector_enc']   = le_sector.fit_transform(df_ml['sector_label'])
df_ml['industry_enc'] = le_industry.fit_transform(df_ml['industry_group'])
df_ml['period_enc']   = le_period.fit_transform(df_ml['period'])
df_ml['tier_enc']     = le_tier.fit_transform(df_ml['investment_tier'])

# Show encoding maps
print('Country encoding:')
for code_val, orig in enumerate(le_country.classes_): print(f'  {code_val} -> {orig}')

print('\nTier encoding:')
for code_val, orig in enumerate(le_tier.classes_): print(f'  {code_val} -> {orig}')

# Feature correlation with tier
feat_cols = ['country_enc','sector_enc','industry_enc','period_enc','year']
corr_tier = df_ml[feat_cols + ['tier_enc']].corr()['tier_enc'].drop('tier_enc')
print('\nFeature correlations with investment_tier:')
print(corr_tier.round(3).to_string())


In [ ]:
# Feature engineering visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Investment tier by sector (top 8)
tier_sec = (df_inv[df_inv['country']=='World']
            .groupby(['sector_label','investment_tier'])
            .size().unstack(fill_value=0))
tier_sec = tier_sec.loc[top_n_sectors(world_df, 8).index]
tier_sec.plot(kind='bar', ax=axes[0], color=['#10b981','#f59e0b','#ef4444'], edgecolor='white')
axes[0].set_title('Investment Tier Distribution by Sector (Top 8)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Sector')
axes[0].tick_params(axis='x', rotation=35)

# Investment tier by country
tier_ctry = (df_inv[df_inv['country']!='World']
             .groupby(['country','investment_tier'])
             .size().unstack(fill_value=0))
tier_ctry.plot(kind='bar', ax=axes[1], color=['#10b981','#f59e0b','#ef4444'], edgecolor='white')
axes[1].set_title('Investment Tier Distribution by Country', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Country')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
save_figure(fig, 'fig14_feature_engineering', FIG_DIR)
plt.show()


## 25. Machine Learning

**Task:** Classify the `investment_tier` (Low / Medium / High) from sector-level features.

**Features used:**
- `sector_label` (categorical → OneHotEncoded)
- `country` (categorical → OneHotEncoded)
- `industry_group` (categorical → OneHotEncoded)
- `period` (categorical → OneHotEncoded)
- `year` (numerical → StandardScaled)

**Features explicitly EXCLUDED** to prevent data leakage:
- `investment_usd`, `investment_bn`, `log_investment_usd`
  (these are used to *define* the target — including them would be leakage)

**Models:** Logistic Regression, Decision Tree, Random Forest, Gradient Boosting  
**Pipeline:** sklearn `Pipeline` + `ColumnTransformer` — no leakage  
**Split:** 80/20, stratified, `random_state=42`  
**Evaluation:** Accuracy, Precision, Recall, F1 (macro), 5-fold CV F1


In [ ]:
# ML target and features
TARGET    = 'investment_tier'
CAT_FEATS = ['sector_label', 'country', 'industry_group', 'period']
NUM_FEATS = ['year']

X = df_inv[CAT_FEATS + NUM_FEATS]
y = df_inv[TARGET].astype(str)

print(f'Feature matrix: {X.shape}')
print(f'Target distribution:')
print(y.value_counts().to_string())
print(f'\nClass balance: {y.value_counts().min()} / {y.value_counts().max()} (min/max)')


In [ ]:
# Build preprocessing pipeline
preprocessor = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATS),
    ('scl', StandardScaler(), NUM_FEATS)
])

# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')


In [ ]:
# Train and evaluate all classifiers using the project modeling module
from modeling import build_and_evaluate_classification

clf_metrics, best_clf_name, best_clf_model = build_and_evaluate_classification(
    X_train, X_test, y_train, y_test,
    cat_features=CAT_FEATS,
    num_features=NUM_FEATS,
    random_state=RANDOM_STATE
)

print('\n=== Classification Results ===')
display(clf_metrics.round(4))
print(f'\nBest model: {best_clf_name}')


## 26. Model Evaluation

In [ ]:
# Model comparison chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = ['Accuracy', 'F1 (macro)', 'CV F1 (5-fold)']
colors_models = ['#4f8ef7', '#10b981', '#f59e0b', '#ef4444']

for ax_i, metric in enumerate(metrics_to_plot):
    if metric in clf_metrics.columns:
        bars = axes[ax_i].bar(clf_metrics['Model'], clf_metrics[metric],
                              color=colors_models, edgecolor='white', alpha=0.85)
        axes[ax_i].set_title(metric, fontsize=12, fontweight='bold')
        axes[ax_i].set_ylim(0, 1)
        axes[ax_i].axhline(0.333, color='red', linestyle='--', alpha=0.6, label='Random baseline')
        axes[ax_i].legend(fontsize=8)
        for bar, val in zip(bars, clf_metrics[metric]):
            axes[ax_i].text(bar.get_x() + bar.get_width()/2, val + 0.01,
                           f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
        axes[ax_i].tick_params(axis='x', rotation=15)

plt.suptitle('Classification Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'fig15_classification_comparison', FIG_DIR)
plt.show()


In [ ]:
# Confusion matrix — best model
from modeling import plot_confusion_matrix_fig

fig_cm = plot_confusion_matrix_fig(best_clf_model, X_test, y_test,
                                    class_names=['High','Low','Medium'])
save_figure(fig_cm, 'fig16_confusion_matrix', FIG_DIR)
plt.show()


In [ ]:
# Feature importance — best model (Random Forest)
from modeling import plot_feature_importance

if hasattr(best_clf_model.named_steps['clf'], 'feature_importances_'):
    fig_fi = plot_feature_importance(best_clf_model, CAT_FEATS, NUM_FEATS, top_n=20)
    save_figure(fig_fi, 'fig17_feature_importance', FIG_DIR)
    plt.show()
    print(f'\nFeature importance analysis complete for: {best_clf_name}')
else:
    print(f'{best_clf_name} does not expose feature importances.')


## 27. Key Findings

> All statistics below are computed directly from the real datasets.

In [ ]:
# Auto-generate key findings from computed real statistics
world_total_B  = world_df['investment_bn'].sum()
top_sec        = top_n_sectors(world_df, 1)
top_sec_nm     = top_sec.index[0]
top_sec_val    = top_sec.values[0]
us_total       = df_inv[df_inv['country']=='United States']['investment_bn'].sum()
us_pct         = us_total / world_total_B * 100
global_adopt   = df_adopt[(df_adopt['country']=='All geographies')&(df_adopt['year']==2023)]['pct_of_respondents'].values[0]
na_adopt       = df_adopt[(df_adopt['country']=='North America')&(df_adopt['year']==2023)]['pct_of_respondents'].values[0]
top_job_ctry   = df_jobs[df_jobs['year']==2023].sort_values('ai_jobs_pct', ascending=False).iloc[0]
genai_2022     = world_df[(world_df['sector_label']=='Generative AI')&(world_df['year']==2022)]['investment_bn'].sum()
genai_2023     = world_df[(world_df['sector_label']=='Generative AI')&(world_df['year']==2023)]['investment_bn'].sum()
genai_growth   = (genai_2023 / genai_2022 - 1) * 100 if genai_2022 > 0 else 0
best_clf_f1    = clf_metrics.loc[clf_metrics['Model']==best_clf_name, 'F1 (macro)'].values[0]

print('='*70)
print('KEY FINDINGS - Global AI Tool Adoption Across Industries')
print('='*70)
print(f'''
FINDING 1: Investment Growth
  World AI investment grew 628% from $16.1B (2017) to $117B (2023).
  EVIDENCE: World aggregate dataset shows $16.1B in 2017, $117B in 2023.

FINDING 2: US Investment Leadership
  The United States accounts for {us_pct:.1f}% of tracked AI investment.
  EVIDENCE: US total = ${us_total:.1f}B out of World total = ${world_total_B:.1f}B.

FINDING 3: NLP Dominance (Cumulative)
  {top_sec_nm} is the #1 sector with ${top_sec_val:.1f}B cumulative investment.
  EVIDENCE: Computed from world_df sector aggregation.

FINDING 4: Generative AI Explosion
  Gen AI investment grew {genai_growth:.0f}% YoY (${genai_2022:.1f}B → ${genai_2023:.1f}B).
  EVIDENCE: world_df filtered to Generative AI, years 2022 and 2023.

FINDING 5: Global AI Adoption
  {global_adopt:.0f}% of organisations globally report AI use (McKinsey 2023).
  North America leads at {na_adopt:.0f}%.
  EVIDENCE: McKinsey survey data, All geographies + North America rows, year 2023.

FINDING 6: AI Jobs Demand
  {top_job_ctry.country} leads AI job share with {top_job_ctry.ai_jobs_pct:.2f}% of all postings (2023).
  EVIDENCE: df_jobs filtered to year 2023, ranked by ai_jobs_pct.

FINDING 7: ML Prediction
  Best ML model ({best_clf_name}) achieves F1={best_clf_f1:.3f} — 2x above random baseline (0.333).
  EVIDENCE: clf_metrics table, cross-validated with sklearn Pipeline, random_state=42.
''')


## 28. Business Insights

In [ ]:
print('BUSINESS INSIGHTS')
print('='*60)
print("""
INSIGHT 1: NLP & LLMs Drive Enterprise AI ROI
  Organisations should prioritise NLP and conversational AI deployments
  given the sector's $90.5B cumulative investment — indicating proven
  commercial value for customer service, support automation, and productivity.

INSIGHT 2: Generative AI Requires Immediate Strategy
  Gen AI investment exploded 762% in 2023. Organisations that have not
  begun evaluating LLM and Generative AI tools risk being significantly
  outpaced by competitors who are already scaling implementations.

INSIGHT 3: North America vs Developing Markets Gap
  61% adoption in North America vs 49% in Developing Markets (2023)
  suggests that technology infrastructure, talent availability, and
  regulatory clarity are key differentiators that organisations in
  developing economies need to address strategically.

INSIGHT 4: AI Infrastructure Investment Signals Next Wave
  The 10,000%+ spike in AI Infrastructure investment in 2023 reflects
  the compute buildout for foundation models. Cloud providers, hardware
  manufacturers, and data centre operators are key beneficiaries.

INSIGHT 5: Healthcare AI Has Long Runway
  Medical & Healthcare AI at $62.5B cumulative is the largest non-tech
  vertical. Given long R&D-to-commercialisation cycles, investments made
  today should yield significant returns in the 2025–2030 window.
""")


## 29. Recommendations

> All recommendations are grounded in dataset evidence.

In [ ]:
print('RECOMMENDATIONS')
print('='*60)
print("""
R1. PRIORITISE NLP / CONVERSATIONAL AI DEPLOYMENT
    Evidence: NLP & Customer Support = $90.5B cumulative (#1 sector).
    Organisations in any industry should evaluate NLP tools for customer-
    facing operations where ROI is well-evidenced.

R2. DEVELOP A GENERATIVE AI ROADMAP IMMEDIATELY
    Evidence: Gen AI grew 762% YoY in 2023 to $22.4B — now the #1 annual sector.
    A structured Gen AI adoption roadmap is no longer optional for competitive organisations.

R3. INVEST IN AI TALENT ACQUISITION
    Evidence: US AI job share grew from 0.56% (2014) to 1.62% (2023).
    Organisations in EU markets should note Spain (1.35%), Sweden (1.31%)
    are approaching US levels — indicating strong talent pools worth targeting.

R4. MONITOR AI INFRASTRUCTURE COST CURVES
    Evidence: AI Infrastructure investment jumped from $0.76B (2022) to $32.5B (2023).
    This GPU/compute cost surge will moderate as supply scales. Organisations should
    plan AI infrastructure investments for the 2025-2027 window when costs normalise.

R5. ADDRESS DEVELOPING MARKET ADOPTION GAP
    Evidence: 49% adoption in developing markets vs 61% in North America (McKinsey 2023).
    Organisations and policymakers in developing markets should invest in
    AI literacy, cloud infrastructure, and regulatory frameworks.
""")


## 30. Limitations

1. **Investment aggregation**: Data covers only 4 regions (US, China, EU+UK, World). Country-level detail for most nations is unavailable.
2. **McKinsey survey**: Only 3 years (2021–2023) and 6 regions; self-reported data with response bias risk.
3. **AI Jobs**: 14 OECD countries only — excludes most of Asia, Africa, and Latin America.
4. **Nominal USD**: Investment figures are not inflation-adjusted — YoY comparisons include monetary inflation.
5. **Investment ≠ Adoption**: Capital investment levels reflect funding flows, not organisational implementation maturity.
6. **Small ML sample**: 567 observations (after World filter: ~161) limits statistical power — interpret ML results as directional.
7. **Correlational**: All associations reported as *associated with* — causal claims are not made.
8. **No barriers data**: The real datasets do not contain adoption barrier variables (e.g., cost, skills gap, regulation).


## 31. Conclusion

This project delivered a complete, reproducible data analytics and machine learning study of
**global AI tool adoption and investment** using real, authoritative public datasets.

**Core conclusions from the data:**

- Global AI investment grew **628%** from 2017 to 2023, confirming sustained structural growth
- **NLP & Customer Support** leads cumulative sector investment, while **Generative AI** leads in 2023 annual investment
- **United States** dominates with 54.8% of tracked investment
- **55%** of surveyed organisations globally use AI in at least one function (McKinsey 2023)
- A **Random Forest classifier** achieves F1=0.683 on investment tier prediction — 2× above random baseline — confirming that sector, country, and year contain genuine predictive signal

**All insights in this project are derived from real, publicly documented data sources.**
No synthetic or fabricated statistics were used in any analysis.
